# CSIRO Biomass image inference

Generate `submission.csv` from a locally trained model uploaded to a Kaggle Dataset.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

COMP_DIR = "/kaggle/input/csiro-biomass"
MODEL_DIR = "/kaggle/input/rikuter67/models"
MODEL_PATH = os.path.join(MODEL_DIR, "image_custom_resnet34.pt")
OUTPUT_PATH = "/kaggle/working/submission.csv"

SRC_ROOT = "/kaggle/input/rikuter67"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from src.models_image import create_model as create_custom_model, is_custom_model


In [ ]:
test_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Missing model at {MODEL_PATH}. Upload your models dataset.")

artifact = torch.load(MODEL_PATH, map_location="cpu")
targets = artifact["targets_order"]
model_name = artifact["model_name"]
image_size = int(artifact.get("image_size", 224))
norm = artifact.get("normalize", {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]})

print("Model:", model_name, "image_size:", image_size)

In [ ]:
class ImageOnlyDataset(Dataset):
    def __init__(self, df, root_dir, tfms):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        image = Image.open(img_path).convert("RGB")
        image = self.tfms(image)
        return image, row["image_path"]


def build_model(name, num_outputs):
    if is_custom_model(name):
        return create_custom_model(name, num_outputs=num_outputs)
    if name == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
        return model
    if name == "resnet34":
        model = models.resnet34(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
        return model
    raise ValueError(f"Unsupported model_name: {name}")


tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm["mean"], std=norm["std"]),
])

unique_images = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
ds = ImageOnlyDataset(unique_images, COMP_DIR, tfms)
loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(model_name, num_outputs=len(targets)).to(device)
model.load_state_dict(artifact["model_state"])
model.eval()

image_to_pred = {}
with torch.no_grad():
    for imgs, paths in loader:
        imgs = imgs.to(device)
        preds = model(imgs).cpu().numpy()
        for p, pred in zip(paths, preds):
            image_to_pred[p] = pred

pred_vals = []
submission = test_df[["sample_id"]].copy()
for _, row in test_df.iterrows():
    img_path = row["image_path"]
    tname = row["target_name"]
    idx = targets.index(tname)
    pred_vals.append(float(image_to_pred[img_path][idx]))

submission["target"] = pred_vals
submission.to_csv(OUTPUT_PATH, index=False)
print("Saved", OUTPUT_PATH)
submission.head()
